#HPC Lab Assignment 7
> Name: Kalpak Gupte \
> Roll No: 44, Panel H \
> PRN: 1032222808 \
> Final Year Btech CSE \

##Environment Setup & GPU Verification

In [ ]:
# Check if NVIDIA GPU is assigned to this session
!nvidia-smi

Tue Mar 31 09:24:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

# Check if CUDA compiler (NVCC) is available
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


##CUDA Vector Addition
This program performs element-wise addition of two vectors A and B of size N, storing the result in vector C.

In [ ]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>

// CUDA Kernel: This runs on the GPU
__global__ void vectorAdd(const int *a, const int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[#include<iostream>
#include<stdio.h>
using namespace std;

__global__ void max(int* input)
{
	int tid = threadIdx.x;		//
	int step_size = 1;
	int number_of_threads = blockDim.x;

	float aux_size = (float)number_of_threads;

	while (number_of_threads > 0) {
		if (tid < number_of_threads)
		{
			int fst = tid * step_size * 2;
			int snd = fst + step_size;
			if (input[fst] < input[snd] && input[snd] > 0)
			{
				input[fst] = input[snd];
			}
		}
		step_size = step_size * 2;
		if (number_of_threads!=1)
		{
			aux_size = aux_size / 2;
			number_of_threads = (int)ceil(aux_size);	//not=int(ceil(not/2));
		}
		else {
			number_of_threads=0;
		}
	}
}

int main(int argc, char const *argv[])
{
	int count = 0;
	cout<<"Enter the number of elements";
	cin>>count;
	int size = count * sizeof(int);

	int h[count];

	cout<<"Elements are";
	for (int i = 0; i < count; i++)
	{
		h[i] = i + 1;
		cout<<h[i]<<endl;
	}



	int *d;
	cudaMalloc(&d,size);
	cudaMemcpy(d,h,size,cudaMemcpyHostToDevice);

	max<<<1,(count/2)+1>>> (d);

	int result;
	cudaMemcpy(&result,d,sizeof(int),cudaMemcpyDeviceToHost);

	cout<<"Max is"<<result;

	getchar();
	cudaFree(d);

	return 0;
}i];
    }
}

int main() {
    int n = 5000; // Size of the vectors
    size_t size = n * sizeof(int);

    // Host memory allocation
    int *h_a = (int *)malloc(size);
    int *h_b = (int *)malloc(size);
    int *h_c = (int *)malloc(size);

    // Initialize vectors
    for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = i * 2;
    }

    // Device memory allocation
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // Copy data from Host to Device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Execution Configuration: 256 threads per block
    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    // Launch the Kernel
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // Copy result back to Host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Display sample results
    std::cout << "--- Vector Addition Results (First 10 Elements) ---" << std::endl;
    for (int i = 0; i < 10; i++) {
        std::cout << h_a[i] << " + " << h_b[i] << " = " << h_c[i] << std::endl;
    }

    // Cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}

Writing vector_add.cu


###Compile and run

In [ ]:
# Compile the code
!nvcc vector_add.cu -o vector_add

# Run the compiled binary
!./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
--- Vector Addition Results (First 10 Elements) ---
0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27


##CUDA Finding Maximum Element

In [ ]:
%%writefile cuda_max.cu
#include <iostream>
#include <stdio.h>
#include <math.h>

using namespace std;

// Note: Renamed from 'max' to 'findMax' to avoid conflicts with std::max
__global__ void findMax(int* input, int total_elements)
{
	int tid = threadIdx.x;
	int step_size = 1;
	int number_of_threads = blockDim.x;

	float aux_size = (float)number_of_threads;

	while (number_of_threads > 0) {
		if (tid < number_of_threads)
		{
			int fst = tid * step_size * 2;
			int snd = fst + step_size;

            // Added boundary check to ensure we don't read garbage memory
			if (snd < total_elements && input[fst] < input[snd])
			{
				input[fst] = input[snd];
			}
		}
		step_size = step_size * 2;

        // CRITICAL: You must sync threads in a reduction before the next loop iteration!
        __syncthreads();

		if (number_of_threads != 1)
		{
			aux_size = aux_size / 2;
			number_of_threads = (int)ceil(aux_size);
		}
		else {
			number_of_threads = 0;
		}
	}
}

int main(int argc, char const *argv[])
{
    // Hardcoded for Colab execution instead of using cin
	int count = 16;
	int size = count * sizeof(int);

    // Fixed VLA issue by dynamically allocating memory
	int *h = (int*)malloc(size);

	cout << "Elements are: ";
	for (int i = 0; i < count; i++)
	{
		h[i] = i + 1; // Generates numbers 1 through 16
		cout << h[i] << " ";
	}
    cout << endl;

	int *d;
	cudaMalloc(&d, size);
	cudaMemcpy(d, h, size, cudaMemcpyHostToDevice);

    // Launch kernel
	findMax<<<1, (count/2) + 1>>>(d, count);

	int result;
    // The max value bubbles up to the 0th index in your logic
	cudaMemcpy(&result, &d[0], sizeof(int), cudaMemcpyDeviceToHost);

	cout << "Max is: " << result << endl;

	cudaFree(d);
    free(h);

	return 0;
}

Overwriting cuda_max.cu


In [ ]:
# Compile the code
!nvcc cuda_max.cu -o cuda_max

# Run the compiled binary
!./cuda_max

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Elements are: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
Max is: 16
